本代码来自：生信技能书  
https://cloud.tencent.com/developer/article/2228252

In [1]:
import os,sys
os.getcwd()
os.listdir(os.getcwd()) 

['Python版SCENIC转录因子分析.ipynb',
 'Seurat_4.3.0.tar.gz',
 'pyscenic转录因子分析.R',
 '.ipynb_checkpoints',
 'pbmc3k.SeuratData_3.1.4.tar.gz',
 'SeuratObject_4.1.3.tar.gz',
 'data',
 'for.scenic.data.csv']

In [2]:
import loompy as lp;
import numpy as np;
import scanpy as sc;

In [3]:
x=sc.read_csv("for.scenic.data.csv");
row_attrs = {"Gene": np.array(x.var_names),};
col_attrs = {"CellID": np.array(x.obs_names)};
lp.create("sample.loom",x.X.transpose(),row_attrs,col_attrs);

**运行pySCENIC**

In [10]:
##运行pySCENIC
# 不同物种的数据库不一样，这里是人类是human 
dir=/disk1/cai026/pyscenic/data/#改成自己的目录
tfs=$dir/hs_hgnc_tfs.txt
feather=$dir/hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings.feather
tbl=$dir/motifs-v9-nr.hgnc-m0.001-o0.0.tbl 
# 一定要保证上面的数据库文件完整无误哦 
input_loom=./sample.loom
ls $tfs  $feather  $tbl  


#因为教程里面的是Bash 脚本  所以要转换为 Python 代码（~）

SyntaxError: invalid syntax (3607864592.py, line 3)

In [11]:
# 定义目录路径
dir_path = "/disk1/cai026/pyscenic/data/"  # 改成自己的目录

# 定义文件路径
tfs = os.path.join(dir_path, "hs_hgnc_tfs.txt")
feather = os.path.join(dir_path, "hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings.feather")
tbl = os.path.join(dir_path, "motifs-v9-nr.hgnc-m0.001-o0.0.tbl")

# 输入 LOOM 文件路径
input_loom = "./sample.loom"

# 检查文件是否存在
files_to_check = [tfs, feather, tbl]

for file_path in files_to_check:
    if not os.path.exists(file_path):
        print(f"文件不存在: {file_path}")
    else:
        print(f"文件存在: {file_path}")

文件存在: /disk1/cai026/pyscenic/data/hs_hgnc_tfs.txt
文件存在: /disk1/cai026/pyscenic/data/hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings.feather
文件存在: /disk1/cai026/pyscenic/data/motifs-v9-nr.hgnc-m0.001-o0.0.tbl


In [18]:
!which pyscenic

**pyscenic的3个步骤之grn**  
构建基因调控网络  
GRN 推断的目的是从单细胞 RNA 测序（scRNA-seq）数据中推断基因调控网络，识别哪些转录因子（TFs）可能调控哪些靶基因。

原理：  
共表达网络构建：首先，根据基因表达数据构建基因共表达网络。高度共表达的基因可能受到相同的调控机制控制。  
转录因子靶标预测：结合已知的转录因子结合位点（TFBS）数据和基因表达数据，预测哪些转录因子可能调节哪些基因。  
调节得分计算：基于共表达网络和转录因子靶标预测，计算每个转录因子对其潜在靶标的调节得分。

In [19]:
#2.1 grn 
!/disk1/cai026/anaconda3/envs/pyscenic/bin/pyscenic grn \
--num_workers 10 \
--output adj.sample.tsv \
--method grnboost2 \
sample.loom \
$tfs #转录因子文件，1839个基因的名字列表


2024-10-23 10:36:19,834 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2024-10-23 10:36:20,019 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.
/disk1/cai026/anaconda3/envs/pyscenic/lib/python3.7/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33847 instead
  f"Port {expected} is already in use.\n"
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child 

**pyscenic的3个步骤之cistarget**  
cisTarget 分析的目的是识别与特定基因组区域（如启动子或增强子）相关的转录因子结合位点（TFBS），从而推断这些区域可能受哪些转录因子调控。

原理:  
motif 富集分析：在特定的基因组区域（如启动子或增强子）中搜索已知的转录因子结合位点（TFBS）。  
统计显著性评估：通过统计方法评估 TFBS 在目标区域中的富集程度，筛选出显著富集的转录因子。

In [20]:
#2.2 cistarget
!/disk1/cai026/anaconda3/envs/pyscenic/bin/pyscenic ctx \
adj.sample.tsv $feather \
--annotations_fname $tbl \
--expression_mtx_fname $input_loom  \
--mode "dask_multiprocessing" \
--output reg.csv \
--num_workers 20  \
--mask_dropouts


2024-10-23 10:42:12,893 - pyscenic.cli.pyscenic - INFO - Creating modules.

2024-10-23 10:42:13,141 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2024-10-23 10:42:13,390 - pyscenic.utils - INFO - Calculating Pearson correlations.

2024-10-23 10:42:13,471 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [True].

2024-10-23 10:42:15,938 - pyscenic.utils - INFO - Creating modules.

2024-10-23 10:42:33,706 - pyscenic.cli.pyscenic - IN

[####                                    ] | 10% Completed |  3min 46.3s
2024-10-23 10:46:20,453 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for PATZ1 could be mapped to hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings. Skipping this module.
[####                                    ] | 10% Completed |  3min 55.4s
2024-10-23 10:46:29,593 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for NELFB could be mapped to hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings. Skipping this module.
[####                                    ] | 10% Completed |  4min 11.1s
2024-10-23 10:46:45,255 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for RREB1 could be mapped to hg19-tss-centered-10kb-10species.mc9nr.genes_vs_motifs.rankings. Skipping this module.
[####                                    ] | 10% Completed |  4min 34.9s
2024-10-23 10:47:09,010 - pyscenic.transform - WARNING - Less than 80% of t

**pyscenic的3个步骤之AUCell**  
AUCell 分析的目的是评估每个细胞中特定基因集的活性水平，从而识别哪些细胞类型或亚群在特定基因集的调控下表现出活性。

原理:  
基因集活性评分：基于基因表达数据，计算每个细胞中特定基因集的活性评分。  
统计显著性评估：通过统计方法评估每个细胞中基因集的活性水平，识别活性显著的细胞。

In [21]:
#2.3 AUCell
!/disk1/cai026/anaconda3/envs/pyscenic/bin/pyscenic aucell \
$input_loom \
reg.csv \
--output out_SCENIC.loom \
--num_workers 10


2024-10-23 10:49:26,848 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2024-10-23 10:49:27,040 - pyscenic.cli.pyscenic - INFO - Loading gene signatures.
Create regulons from a dataframe of enriched features.
Additional columns saved: []

2024-10-23 10:49:28,210 - pyscenic.cli.pyscenic - INFO - Calculating cellular enrichment.

2024-10-23 10:49:30,911 - pyscenic.cli.pyscenic - INFO - Writing results to file.


**out_SCENIC.loom导入R里面进行可视化**